Week 14 · Day 1 — Why Fine-Tune LLMs?
Why this matters

Pretrained LLMs are powerful, but they’re generalists. Fine-tuning lets you adapt them to your domain (e.g., legal, medical, customer support) or task (e.g., classification, summarization) while saving compute. Knowing how much of the model to tune is key for efficiency.

Theory Essentials

Full fine-tuning: update all weights; high accuracy but expensive (billions of params).

Partial fine-tuning: only train last layers/classifier head; cheap but limited flexibility.

Adapter tuning / PEFT (Parameter-Efficient Fine-Tuning): inject small trainable modules (like LoRA) while freezing most weights.

LoRA: trains low-rank matrices inside attention layers → huge memory savings.

Tradeoff: accuracy vs compute vs deployment size.

This shows zero-shot inference before fine-tuning.

Zero-shot inference means:
You take a pretrained model (like DistilBERT) and run it directly on your text without doing any fine-tuning on your specific task dataset.

In [6]:
# Setup
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load base model (pretrained on general text)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Example text
text = "This movie was surprisingly good!"
inputs = tokenizer(text, return_tensors="pt")

# Forward pass (no fine-tuning yet)
with torch.no_grad():
    outputs = model(**inputs)

print("Logits:", outputs.logits)
print("Predicted class:", torch.argmax(outputs.logits, dim=1).item())


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Logits: tensor([[-0.1005,  0.1036]])
Predicted class: 1


1) Core (10–15 min)
Task: Try 3 different sentences (positive/negative) and see how the pretrained model classifies them.

In [7]:
for text in ["I hated this film", "Absolutely wonderful!", "Mediocre and boring"]:
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
    print(text, "→", torch.argmax(logits, dim=1).item())


I hated this film → 1
Absolutely wonderful! → 1
Mediocre and boring → 1


Not very good at predicting right now. 

2) Practice (10–15 min)
Task: Replace distilbert-base-uncased with bert-base-uncased. Compare output logits and predicted classes.


In [ ]:
model_name2 = "bert-base-uncased"
tokenizer2 = AutoTokenizer.from_pretrained(model_name2)
model2 = AutoModelForSequenceClassification.from_pretrained(model_name2, num_labels=2)

for text in ["I hated this film", "Absolutely wonderful!", "Mediocre and boring"]:
    inputs = tokenizer2(text, return_tensors="pt")
    with torch.no_grad():
        logits = model2(**inputs).logits
    print(text, "→", torch.argmax(logits, dim=1).item())

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


I hated this film → 0
Absolutely wonderful! → 0
Mediocre and boring → 0


3) Stretch (optional, 10–15 min)
Task: Print the number of trainable parameters for the model.

In [9]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / Total: {total:,} ({trainable/total:.2%})")


Trainable: 109,483,778 / Total: 109,483,778 (100.00%)


Mini-Challenge (≤40 min)

Fine-tuning often means freezing most layers.
Task: Freeze all layers except the classifier head and print how many parameters remain trainable.
Acceptance Criteria: Only ~0.5–1% of parameters should remain trainable.

In [10]:
# Mini-Challenge: freeze all but the classifier head

from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 1) Freeze the backbone
# (works for BERT/DistilBERT/RoBERTa via .base_model)
for p in model.base_model.parameters():
    p.requires_grad = False

# 2) Unfreeze only the classifier head (and pre_classifier if it exists)
to_unfreeze = []
if hasattr(model, "classifier"):
    to_unfreeze.append(model.classifier)
if hasattr(model, "pre_classifier"):          # DistilBERT has this linear layer before classifier
    to_unfreeze.append(model.pre_classifier)

for module in to_unfreeze:
    for p in module.parameters():
        p.requires_grad = True

# 3) Report counts
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / Total: {total:,}  ({trainable/total:.2%})")

# (Optional) Show which params are trainable
print("\nTrainable parameter tensors:")
for n, p in model.named_parameters():
    if p.requires_grad:
        print(f"  {n:50s}  -> {p.numel():,} params")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable: 592,130 / Total: 66,955,010  (0.88%)

Trainable parameter tensors:
  pre_classifier.weight                               -> 589,824 params
  pre_classifier.bias                                 -> 768 params
  classifier.weight                                   -> 1,536 params
  classifier.bias                                     -> 2 params


Notes / Key Takeaways

Fine-tuning adapts a general model to a specific use case.

Full fine-tune = best accuracy but very costly.

Partial fine-tune = lightweight but less flexible.

PEFT (LoRA, adapters) = balance between efficiency and performance.

Hugging Face makes it easy to load, freeze, and extend models.

Reflection

Why might full fine-tuning be unnecessary for most real-world tasks?

How does freezing parameters reduce cost but also limit adaptation?

1) Why might full fine-tuning be unnecessary for most real-world tasks?
Pretrained LLMs already capture general language patterns. For many tasks (like sentiment, topic, or intent classification), adapting only the classifier head or a small subset of layers is enough — no need to retrain the entire model. This saves compute and data while still reaching high accuracy.

2) How does freezing parameters reduce cost but also limit adaptation?
Freezing cuts training cost because gradients don’t need to be stored or updated for most of the model (less memory and compute). But it also limits flexibility — the model can’t deeply adjust to domain-specific language, so performance may suffer on tasks requiring more nuanced adaptation.